In [178]:
import os
import glob
import pandas as pd
import warnings

In [339]:
# Path to folder containing .xlsx files
DATA_DIR = "./data/related/outcomes_endpoints"


In [341]:
# Required columns in each Excel file
REQUIRED_COLS = [
    'Test Name',
    'Measure',
    'Outcome Domain',
    'Subdomain'
]

# Pattern to match .xlsx files
file_pattern = os.path.join(DATA_DIR, '*.xlsx')
excel_files = glob.glob(file_pattern)

if not excel_files:
    warnings.warn(f"No Excel files found in {DATA_DIR}")

# List to collect dataframes
dfs = []

for filepath in excel_files:
    if 'harmonized' in filepath:
        continue
    # Read the Excel file into a DataFrame
    df = pd.read_excel(filepath)

    filename = os.path.basename(filepath)

    # Check for required columns
    missing = set(REQUIRED_COLS) - set(df.columns)
    if missing:
        warnings.warn(
            f"File '{filename}' is missing columns: {', '.join(missing)}",
            UserWarning
        )
        # Continue processing; missing columns will result in NaNs

    # Normalize Test Name: lowercase and strip whitespace
    if 'Test Name' in df.columns:
            df['Test Name'] = (
                df['Test Name']
                .astype(str)
                .str.strip()
                .str.lower()
                # Normalize dashes to standard hyphen
                .str.replace(r'[‐–—]', '-', regex=True)
                # Remove trailing words 'test', 'tests', 'task', or 'tasks'
                #.str.replace(r"\s+(?:test|tests|task|tasks)$", '', regex=True)
            )


    # Extract article key from filename (e.g. Endpoints_shepherdTranslationalAssaysAssessment2016)
    article_key = os.path.splitext(filename)[0].split('_')[-1]

    # Add a column for the source article
    df['Article'] = article_key

    dfs.append(df)

# Concatenate all into one DataFrame
combined = pd.concat(dfs, ignore_index=True)

# Define aggregation functions:
# - For Measure: join unique values
# - Outcome Domain, Subdomain: join unique values
# - Articles: join unique values
agg_funcs = {
    'Measure': lambda x: '; '.join(sorted(set(x.dropna()))),
    'Outcome Domain': lambda x: '; '.join(sorted(set(x.dropna()))),
    'Subdomain': lambda x: '; '.join(sorted(set(x.dropna()))),
    'Article': lambda x: '; '.join(sorted(set(x.dropna())))
}

# Group by normalized Test Name and aggregate
harmonized_raw = combined.groupby('Test Name', as_index=False).agg(agg_funcs)

# Optional: reorder columns
columns = ['Test Name', 'Measure', 'Outcome Domain', 'Subdomain', 'Article']
harmonized_raw = combined.groupby('Test Name', as_index=False).agg(agg_funcs)

# Save to Excel or CSV
output_path = os.path.join(DATA_DIR, 'harmonized_assays_raw.xlsx')
harmonized.to_excel(output_path, index=False)

print(f"Harmonized DataFrame saved to {output_path}")


Harmonized DataFrame saved to ./data/related/outcomes_endpoints/harmonized_assays_raw.xlsx


/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_25074/1052723272.py:30: UserWarning: File 'Endpoints_osierChronicHistopathologicalBehavioral2015.xlsx' is missing columns: Test Name
  warnings.warn(
/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_25074/1052723272.py:30: UserWarning: File 'Endpoints_waerzeggersMouseModelsNeurological2010.xlsx' is missing columns: Subdomain
  warnings.warn(
/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_25074/1052723272.py:30: UserWarning: File 'Enpoints_emborgEvaluationAnimalModels2004.xlsx' is missing columns: Subdomain
  warnings.warn(
/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_25074/1052723272.py:30: UserWarning: File 'Endpoints_meredithBehavioralModelsParkinsons2006.xlsx' is missing columns: Subdomain
  warnings.warn(
/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_25074/1052723272.py:30: UserWarning: File 'Endpoints_markicevicEmergingImagingMethods2021a.xlsx' is missing columns: Subdomain


In [343]:
harmonized_raw 


,Test Name,Measure,Outcome Domain,Subdomain,Article
0,3-chamber test,Crosses into empty; Crosses into occupied; Lat...,Behavioral,Sociability,harrisonUnifiedBehavioralScoring2020
1,5-choice serial reaction time task (5-csrtt),"Response accuracy, omissions, premature respon...",Behavioral,Attention,shepherdTranslationalAssaysAssessment2016
2,5-choice serial reaction time task (csrtt),"Accuracy, omissions, reaction time, premature ...",Behavioral,Attention & executive function,websterUsingMiceModel2014
3,[^11c]flumazenil pet,GABAA receptor binding potential,Molecular & Cellular,,waerzeggersMouseModelsNeurological2010
4,[^11c]methionine (met) pet,Amino‐acid transport/protein synthesis (tumor ...,Physiological & Metabolic,,waerzeggersMouseModelsNeurological2010
...,...,...,...,...,...
322,wrist flexion-extension handle task,Range and speed of wrist movements when operat...,Behavioral,,emborgEvaluationAnimalModels2004
323,writhing test (acetic acid),Number of abdominal constrictions (“writhes”),Behavioral,Visceral pain,gregoryOverviewAnimalModels2013; sadlerInnovat...
324,y-maze,% spontaneous alternation among the three arms...,Behavioral,Working memory,goldFunctionalAssessmentLongTerm2013; shepherd...
325,y-maze alternation,% spontaneous alternation (entries into novel ...,Behavioral,Working memory,wahlCognitiveBehavioralEvaluation2017


In [345]:
set(harmonized_raw['Outcome Domain'])

{'Behavioral',
 'Imaging & Morphology',
 'Imaging & Morphology; Physiological & Metabolic',
 'Molecular & Cellular',
 'Physiological & Metabolic'}

## Harmonize ChatGPT help

In [348]:
mapping = {
    # 5-choice serial reaction time task variants
    '5-choice serial reaction time task (5-csrtt)': '5-choice serial reaction time task',
    '5-choice serial reaction time task (csrtt)':  '5-choice serial reaction time task',

    # Adhesive removal
    'adhesive removal':                           'adhesive removal test',
    'bilateral tactile adhesive removal':         'adhesive removal test',

    # Adjusting‐step test variants
    'adjusting steps':                            'adjusting-step test',
    'adjusting-step (stepping) test':             'adjusting-step test',

    # Beam/Balance-beam variants
    'balance beam':                               'beam walking test',
    'beam traversal':                             'beam walking test',
    'beam walking':                               'beam walking test',
    'beam walking test':                          'beam walking test',

    # BOLD fMRI sequence variants
    'bold fmri (gre-epi)':                        'bold fmri',
    'bold fmri (se-epi)':                         'bold fmri',

    # Conditioned place avoidance ≃ aversion
    'conditioned place avoidance (cpa)':          'conditioned place aversion',

    # Cylinder test variants
    'cylinder (forelimb asymmetry) test':         'cylinder test',
    'cylinder spontaneous placement':             'cylinder test',
    'limb use asymmetry (cylinder)':              'cylinder test',

    # Delayed/non-matching to place water maze
    'delay-nonmatching-to-place (dnmtp) water maze':  'delayed matching-to-place water maze',
    'delayed matching-to-place (dmp) water maze':      'delayed matching-to-place water maze',
    'delayed matching/non-matching to position (dmtp/dmts)': 'delayed matching-to-place water maze',

    # Diffusion MRI variants
    'diffusion mri (adc mapping)':                'diffusion mri',
    'diffusion mri (dmri)':                       'diffusion mri',

    # DTI vs DWI
    'diffusion tensor imaging (dti)':             'diffusion tensor imaging',
    'diffusion-weighted imaging (dwi)':           'diffusion tensor imaging',

    # Functional MRI variants
    'functional mri (bold fmri)':                 'functional mri',
    'functional mri (fmri)':                      'functional mri',

    # Foot-fault
    'foot fault':                                 'foot fault test',
    'foot-fault test':                            'foot fault test',

    # Forced swim
    'forced swim task':                           'forced swim test',
    'forced swim test':                           'forced swim test',

    # Fear conditioning
    'fear conditioning':                          'fear conditioning test',
    'fear conditioning (general fc)':             'fear conditioning test',
    'fear conditioning - contextual':             'fear conditioning test',
    'fear conditioning - contextual (fc-c)':      'fear conditioning test',
    'fear conditioning - cued':                   'fear conditioning test',
    'fear conditioning - cued (fc-u)':            'fear conditioning test',
    'fear conditioning test':                     'fear conditioning test',

    # Grid test variants
    'grid task (acute mptp mice)':                'grid test',
    'grid-walk test':                             'grid test',

    # Grip strength
    'grip strength meter test':                   'grip strength test',

    # Grooming
    'grooming':                                   'grooming test',

    # Hargreaves
    'hargreaves (radiant-heat) test':             'hargreaves test',

    # Home-cage monitoring
    'home-cage activity monitoring':              'home cage monitoring',

    # Inclined plane
    'inclined plane':                             'inclined plane test',

    # Intrinsic optical imaging
    'intrinsic optical imaging (ioi)':            'intrinsic optical imaging',

    # Laser speckle contrast imaging
    'laser speckle contrast imaging (lsci)':      'laser speckle contrast imaging',

    # Light–dark box
    'light-dark compartment test':                'light-dark box test',
    'light/dark box':                             'light-dark box test',

    # Limb placement
    'limb-placing function':                      'limb placement test',

    # MRI / MRS
    'magnetic resonance imaging (mri)':           'magnetic resonance imaging',
    'magnetic resonance spectroscopy (mrs)':      'magnetic resonance spectroscopy',
    'magnetic resonance spectroscopy mrs (proton magnetic resonance spectroscopy)': 'magnetic resonance spectroscopy',
    'mr spectroscopy (mrs)':                      'magnetic resonance spectroscopy',

    # Modified Neurological Severity Score
    'modified neurological severity score':               'modified neurological severity score',
    'modified neurological severity score (mnss)':        'modified neurological severity score',
    'modified neurological severity score (mnss) (mouse)': 'modified neurological severity score',
    'modified neurological severity score (mnss) (rat)':   'modified neurological severity score',

    # Morris Water Maze variants
    'morris water maze':                         'morris water maze',
    'morris water maze (mwm)':                   'morris water maze',
    'morris water maze - learning latency':      'morris water maze',
    'morris water maze - probe trial':           'morris water maze',
    'morris water maze - reversal learning':     'morris water maze',
    'morris water maze - swim speed':            'morris water maze',
    'morris water maze - working memory':        'morris water maze',

    # Novel Object Recognition
    'novel object recognition':                  'novel object recognition test',
    'novel object recognition (nor)':            'novel object recognition test',
    'novel object recognition test (nort)':      'novel object recognition test',

    # Open-Field / Open Field
    'open field locomotion':                     'open field test',
    'open field test':                           'open field test',
    'open-field':                                'open field test',
    'open-field activity':                       'open field test',
    'open-field test':                           'open field test',

    # Neurological Severity Score (unmodified)
    'neurological severity score (nss)':         'neurological severity score',
    'neurological scoring':                      'neurological severity score',

    # Passive avoidance
    'passive avoidance':                         'passive avoidance test',

    # Perfusion MRI
    'perfusion mri (cbf-weighted fmri)':         'perfusion mri (cbf-weighted)',
    'perfusion mri (cbv-weighted fmri)':         'perfusion mri (cbv-weighted)',

    # Radial-arm maze variants
    'radial arm maze (ram)':                     'radial arm maze',
    'radial-arm maze':                           'radial arm maze',
    'radial arm water maze (rawm)':              'radial arm water maze',
    'radial-arm water maze (rawm)':              'radial arm water maze',

    # Resting-state fMRI
    'resting-state fmri (bold)':                 'resting-state fmri',

    # Rotarod
    'rotarod':                                   'rotarod test',

    # Single-pellet reaching
    'single pellet reaching':                    'single pellet reaching task',

    # Staircase
    'staircase':                                 'staircase test',
    'staircase test (monkey “hill”/“valley”)':   'staircase test',

    # Social approach/avoidance
    'social approach':                           'social approach test',
    'social avoidance':                          'social avoidance test',

    # T-maze variants
    't-maze/y-maze alternation':                 't-maze',

    # T2-weighted MRI variant
    't2/t2-weighted mri*':                      't2-weighted mri',

    # Three-chamber social approach
    'three-chambered social approach test':      'three-chamber social approach test',

    # What-where-which
    'what-where-which (wwwhich) task':           'what-where-which (wwwhich)',

    # Wheel running
    'wheel running activity':                    'wheel running',

    # Von Frey filament
    'von frey filament test (up/down)':          'von frey filament test',

    # Y-maze alternation
    'y-maze alternation':                        'y-maze',
}


In [350]:
# 2) Secondary processing: apply mapping, collect synonyms, and re-group
# Map raw names to canonical
harmonized_raw['Canonical Name'] = harmonized_raw['Test Name'].replace(mapping)
# Keep raw as synonym
harmonized_raw['Synonym'] = harmonized_raw['Test Name']

# Group by Canonical Name
agg2 = {
    'Measure': lambda x: '; '.join(sorted(set(x.dropna()))),
    'Outcome Domain': lambda x: sorted(set(x.dropna()))[0],
    'Subdomain': lambda x: '; '.join(sorted(set(x.dropna()))),
    'Article': lambda x: '; '.join(sorted(set(x.dropna()))),
    'Synonym': lambda x: '; '.join(sorted(set(x.dropna())))
}
final_harmonized = harmonized_raw.groupby('Canonical Name', as_index=False).agg(agg2)

# Reorder columns
cols = ['Canonical Name', 'Synonym', 'Outcome Domain', 'Subdomain', 'Measure', 'Article']
final_harmonized = final_harmonized[cols]

# Save to Excel
output_path = os.path.join(DATA_DIR, 'final_harmonized_with_synonyms.xlsx')
final_harmonized.to_excel(output_path, index=False)
print(f"Saved final harmonized table with synonyms to {output_path}")

Saved final harmonized table with synonyms to ./data/related/outcomes_endpoints/final_harmonized_with_synonyms.xlsx


In [352]:
final_harmonized

,Canonical Name,Synonym,Outcome Domain,Subdomain,Measure,Article
0,3-chamber test,3-chamber test,Behavioral,Sociability,Crosses into empty; Crosses into occupied; Lat...,harrisonUnifiedBehavioralScoring2020
1,5-choice serial reaction time task,5-choice serial reaction time task (5-csrtt); ...,Behavioral,Attention; Attention & executive function,"Accuracy, omissions, reaction time, premature ...",shepherdTranslationalAssaysAssessment2016; web...
2,[^11c]flumazenil pet,[^11c]flumazenil pet,Molecular & Cellular,,GABAA receptor binding potential,waerzeggersMouseModelsNeurological2010
3,[^11c]methionine (met) pet,[^11c]methionine (met) pet,Physiological & Metabolic,,Amino‐acid transport/protein synthesis (tumor ...,waerzeggersMouseModelsNeurological2010
4,[^11c]mp4a pet,[^11c]mp4a pet,Molecular & Cellular,,Acetylcholinesterase activity (tracer trapping...,waerzeggersMouseModelsNeurological2010
...,...,...,...,...,...,...
250,wire grip test,wire grip test,Behavioral,Grip strength & neuromuscular endurance,Score (0–5) based on grasping and movement on ...,goldFunctionalAssessmentLongTerm2013
251,wrist flexion-extension handle task,wrist flexion-extension handle task,Behavioral,,Range and speed of wrist movements when operat...,emborgEvaluationAnimalModels2004
252,writhing test (acetic acid),writhing test (acetic acid),Behavioral,Visceral pain,Number of abdominal constrictions (“writhes”),gregoryOverviewAnimalModels2013; sadlerInnovat...
253,y-maze,y-maze; y-maze alternation,Behavioral,Working memory,% spontaneous alternation (entries into novel ...,goldFunctionalAssessmentLongTerm2013; shepherd...


### add GPT synonyms

In [355]:
df_syn_gpt

,Canonical Name,Synonyms_GPT
0,3-chamber test,three-chamber social approach test; three-cham...
1,5-choice serial reaction time task,5-CSRTT; five-choice serial reaction time para...
2,[^11c]flumazenil pet,[^11C]flumazenil PET;11C-flumazenil PET;[^11C]...
3,[^11c]methionine (met) pet,[^11C]MET PET;C-11 methionine PET;MET-11C PET;...
4,[^11c]mp4a pet,[^11C]MP4A PET;11C-MP4A PET;MP4A-11C PET;[^11C...
...,...,...
250,wire grip test,wire hang test; grip strength wire assay
251,wrist flexion-extension handle task,wrist flexion-extension task; forelimb flexion...
252,writhing test (acetic acid),acetic acid writhing assay; abdominal constric...
253,y-maze,Y-maze; spontaneous alternation Y-maze


In [357]:
df_syn_gpt = pd.read_excel(os.path.join(DATA_DIR, 'harmonized_synonyms_GPT.xlsx'))

final_harmonized = final_harmonized.merge(
    df_syn_gpt[['Canonical Name', 'Synonyms_GPT']],
    on='Canonical Name',
    how='left'
)

# Merge GPT synonyms into final_harmonized
final_harmonized['Synonym'] = final_harmonized.apply(
    lambda row: '; '.join(
        [row['Synonym']] + ([row['Synonyms_GPT']] if pd.notnull(row['Synonyms_GPT']) else [])
    ), axis=1
)
# Drop the temporary Synonyms_GPT column
final_harmonized = final_harmonized.drop(columns=['Synonyms_GPT'])




In [359]:
new_row = {
    'Canonical Name': 'positron emission tomography',
    'Synonym': 'positron emission tomography; PET imaging; metabolic PET',
    'Outcome Domain': 'Physiological & Metabolic',
    'Subdomain': 'Nuclear Imaging',
    'Measure': 'Standard uptake value (SUV); SUV_max; SUV_mean',
    'Article': 'custom'
}

# Add the row
final_harmonized = pd.concat([final_harmonized, pd.DataFrame([new_row])], ignore_index=True)


In [361]:
final_harmonized

,Canonical Name,Synonym,Outcome Domain,Subdomain,Measure,Article
0,3-chamber test,3-chamber test; three-chamber social approach ...,Behavioral,Sociability,Crosses into empty; Crosses into occupied; Lat...,harrisonUnifiedBehavioralScoring2020
1,5-choice serial reaction time task,5-choice serial reaction time task (5-csrtt); ...,Behavioral,Attention; Attention & executive function,"Accuracy, omissions, reaction time, premature ...",shepherdTranslationalAssaysAssessment2016; web...
2,[^11c]flumazenil pet,[^11c]flumazenil pet; [^11C]flumazenil PET;11C...,Molecular & Cellular,,GABAA receptor binding potential,waerzeggersMouseModelsNeurological2010
3,[^11c]methionine (met) pet,[^11c]methionine (met) pet; [^11C]MET PET;C-11...,Physiological & Metabolic,,Amino‐acid transport/protein synthesis (tumor ...,waerzeggersMouseModelsNeurological2010
4,[^11c]mp4a pet,[^11c]mp4a pet; [^11C]MP4A PET;11C-MP4A PET;MP...,Molecular & Cellular,,Acetylcholinesterase activity (tracer trapping...,waerzeggersMouseModelsNeurological2010
...,...,...,...,...,...,...
251,wrist flexion-extension handle task,wrist flexion-extension handle task; wrist fle...,Behavioral,,Range and speed of wrist movements when operat...,emborgEvaluationAnimalModels2004
252,writhing test (acetic acid),writhing test (acetic acid); acetic acid writh...,Behavioral,Visceral pain,Number of abdominal constrictions (“writhes”),gregoryOverviewAnimalModels2013; sadlerInnovat...
253,y-maze,y-maze; y-maze alternation; Y-maze; spontaneou...,Behavioral,Working memory,% spontaneous alternation (entries into novel ...,goldFunctionalAssessmentLongTerm2013; shepherd...
254,zte-cbv fmri,zte-cbv fmri,Imaging & Morphology,,CBV-weighted activation with zero-time-echo SE...,markicevicEmergingImagingMethods2021a


In [363]:
# Save to Excel
output_path = os.path.join(DATA_DIR, 'final_harmonized_with_enriched_synonyms.xlsx')
final_harmonized.to_excel(output_path, index=False)
print(f"Saved enriched harmonized table with synonyms to {output_path}")

Saved enriched harmonized table with synonyms to ./data/related/outcomes_endpoints/final_harmonized_with_enriched_synonyms.xlsx


In [365]:
final_harmonized.to_csv("../08_IE_full_text/data/assay_extraction/assay_final_harmonized_with_enriched_synonyms.csv", index=False)

df_test = pd.read_csv("../08_IE_full_text/data/assay_extraction/assay_final_harmonized_with_enriched_synonyms.csv")

In [367]:
df_test

,Canonical Name,Synonym,Outcome Domain,Subdomain,Measure,Article
0,3-chamber test,3-chamber test; three-chamber social approach ...,Behavioral,Sociability,Crosses into empty; Crosses into occupied; Lat...,harrisonUnifiedBehavioralScoring2020
1,5-choice serial reaction time task,5-choice serial reaction time task (5-csrtt); ...,Behavioral,Attention; Attention & executive function,"Accuracy, omissions, reaction time, premature ...",shepherdTranslationalAssaysAssessment2016; web...
2,[^11c]flumazenil pet,[^11c]flumazenil pet; [^11C]flumazenil PET;11C...,Molecular & Cellular,NaN,GABAA receptor binding potential,waerzeggersMouseModelsNeurological2010
3,[^11c]methionine (met) pet,[^11c]methionine (met) pet; [^11C]MET PET;C-11...,Physiological & Metabolic,NaN,Amino‐acid transport/protein synthesis (tumor ...,waerzeggersMouseModelsNeurological2010
4,[^11c]mp4a pet,[^11c]mp4a pet; [^11C]MP4A PET;11C-MP4A PET;MP...,Molecular & Cellular,NaN,Acetylcholinesterase activity (tracer trapping...,waerzeggersMouseModelsNeurological2010
...,...,...,...,...,...,...
251,wrist flexion-extension handle task,wrist flexion-extension handle task; wrist fle...,Behavioral,NaN,Range and speed of wrist movements when operat...,emborgEvaluationAnimalModels2004
252,writhing test (acetic acid),writhing test (acetic acid); acetic acid writh...,Behavioral,Visceral pain,Number of abdominal constrictions (“writhes”),gregoryOverviewAnimalModels2013; sadlerInnovat...
253,y-maze,y-maze; y-maze alternation; Y-maze; spontaneou...,Behavioral,Working memory,% spontaneous alternation (entries into novel ...,goldFunctionalAssessmentLongTerm2013; shepherd...
254,zte-cbv fmri,zte-cbv fmri,Imaging & Morphology,NaN,CBV-weighted activation with zero-time-echo SE...,markicevicEmergingImagingMethods2021a
